<a href="https://colab.research.google.com/github/Decoding-Data-Science/sharjah/blob/main/notebooks/Day3_Simple_Agent_Gemini_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Decoding-Data-Science/sharjah/blob/main/notebooks/Day3_Simple_Agent_Gemini_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Simple Agent: SQL Tool + RAG Tool

**Workshop: Data and AI with Gemini | Day 3**

On Day 1 you worked with the airport database. On Day 2 you built RAG on two airport PDF documents.
Today we put both together. We give Gemini **two tools** and let it **decide** which one to use.

An **agent** is a model that can choose a tool, use it, read the result, and then answer.

```
                         +--> Tool 1: airport_policy    (RAG on the 2 PDFs)    "What is the RULE?"
 Question --> AGENT -----+
                         +--> Tool 2: airport_database  (SQL on airport_ops.db) "What are the NUMBERS?"
```

## What you will do
1. Install the libraries and connect to Gemini (`gemini-3.5-flash`) with your key
2. Upload the two PDF files and the database
3. Build **Tool 1**: the RAG engine from Day 2, wrapped as a tool
4. Build **Tool 2**: a read-only SQL function, wrapped as a tool
5. Create the **agent** and watch which tool it picks
6. Ask a question that needs **both** tools
7. Put a **Gradio** app on top

Press **Shift + Enter** to run a cell. Run the cells in order.

## Step 1. Install the libraries

This takes about a minute the first time. The second cell prints the versions so you know it worked.

In [1]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-readers-file pypdf gradio

In [2]:
import importlib.metadata as md
for package in ["llama-index-core", "llama-index-llms-google-genai", "llama-index-embeddings-google-genai", "gradio"]:
    print(package, md.version(package))

llama-index-core 0.14.25
llama-index-llms-google-genai 0.11.3
llama-index-embeddings-google-genai 0.6.0
gradio 6.31.0


## Step 2. Connect to Gemini

Your key comes from **Colab Secrets**, never from the notebook.
Click the **key icon** on the left, add a secret named `GEMINI_API_KEY`, paste your key, and switch on **Notebook access**.

We use two Gemini models:
- **`gemini-3.5-flash`** is the agent's brain. It reads the question, picks the tool, and writes the answer.
- **`gemini-embedding-001`** turns text into numbers for the RAG search (768 numbers per text).

`THINKING_LEVEL` tells Gemini how much to think before it answers: `minimal`, `low`, `medium` or `high`. Agents choose tools well at `low`, and it is faster and cheaper.

In [7]:
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
from google.genai import types
from google.genai.types import EmbedContentConfig
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

API_KEY = userdata.get("gemini3")

MODEL = "gemini-3.5-flash"           # if the free tier gives a quota error, use "gemini-3.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"
THINKING_LEVEL = "low"

llm = GoogleGenAI(
    model=MODEL,
    api_key=API_KEY,
    generation_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=THINKING_LEVEL)
    ),
)
Settings.llm = llm
Settings.embed_model = GoogleGenAIEmbedding(
    model_name=EMBED_MODEL,
    api_key=API_KEY,
    embedding_config=EmbedContentConfig(output_dimensionality=768),
)

# Quick test of both models
print(llm.complete("Say hello to the workshop in one short sentence."))
print("The embedding model returned", len(Settings.embed_model.get_text_embedding("departure immigration queue")), "numbers")

Hello everyone, and welcome to the workshop!
The embedding model returned 768 numbers


## Step 3. Upload the two PDFs and the database

Run the next cell, click **Choose files**, and select **three files** in one go:
- `GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf`
- `GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf`
- `airport_ops.db`

In [9]:
import os
from google.colab import files

PDF_FOLDER = "/content/airport_rag"
DB_PATH = "/content/airport_ops.db"
os.makedirs(PDF_FOLDER, exist_ok=True)

print("Select the 2 PDF files and airport_ops.db:")
uploaded = files.upload()

for name, data in uploaded.items():
    path = DB_PATH if name.endswith(".db") else os.path.join(PDF_FOLDER, name)
    with open(path, "wb") as f:
        f.write(data)

Select the 2 PDF files and airport_ops.db:


Saving GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf to GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf
Saving GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf to GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf


In [10]:
print("PDF files :", sorted(os.listdir(PDF_FOLDER)))
print("Database  :", DB_PATH, "|", round(os.path.getsize(DB_PATH) / 1_000_000, 1), "MB")

PDF files : ['GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf', 'GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf']
Database  : /content/airport_ops.db | 29.4 MB


## Step 4. Tool 1: the RAG engine (policy documents)

This is the same RAG you built on Day 2, with the settings you locked in: **sentence chunks of 300 tokens, overlap 30, top_k 3**, and the **rule prompt** that makes the model cite the document and say "I could not find this" when the answer is not there.

First we build and test it **on its own**.

In [11]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, PromptTemplate
from llama_index.core.node_parser import SentenceSplitter

documents = SimpleDirectoryReader(input_dir=PDF_FOLDER, required_exts=[".pdf"]).load_data()

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=300, chunk_overlap=30)],
)
print(len(documents), "pages loaded,", len(index.docstore.docs), "chunks in the index")

/usr/lib/python3.13/textwrap.py:439: RuntimeWarning: coroutine 'prepare_chat_params' was never awaited
  margin = indent


3 pages loaded, 8 chunks in the index


In [13]:
qa_prompt = PromptTemplate(
    "You are the Airport Operations Assistant.\n"
    "Answer using ONLY the context below.\n"
    "Cite the document number and section for every rule, for example [GIA-POL-01, 2.3].\n"
    "If the context does not contain the answer, say exactly: I could not find this in the documents.\n"
    "Keep the answer under 120 words.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer: "
)

policy_engine = index.as_query_engine(similarity_top_k=3, text_qa_template=qa_prompt)

print(policy_engine.query("What is the maximum average wait for departure immigration?"))

The maximum average wait for departure immigration must not exceed 15 minutes [GIA-POL-01, 2.3].


### Wrap the RAG engine as a tool

A tool is: **a name, a description, and something that runs**. The agent never sees our code. It only reads the **name and description** to decide when to use the tool. So write the description carefully.

In [14]:
from llama_index.core.tools import QueryEngineTool

policy_tool = QueryEngineTool.from_defaults(
    query_engine=policy_engine,
    name="airport_policy",
    description=(
        "Use this tool for questions about airport RULES: service standards, waiting-time limits, "
        "minimum number of counters, escalation levels, what the duty manager must do, smart gates. "
        "Sources: GIA-POL-01 Passenger Service Standards Policy and GIA-SOP-06 Peak-Period Operations and Queue Escalation SOP. "
        "Input: a plain-language question. Output: an answer with document number and section."
    ),
)
print("Tool ready:", policy_tool.metadata.name)

Tool ready: airport_policy


## Step 5. Tool 2: the SQL tool (airport database)

Now the numbers. We write a normal Python function that runs a SQL query and returns the result as text. Three safety rules are built in:
1. The database is opened **read-only**
2. Only a single **SELECT** query is allowed
3. At most **30 rows** are sent back to the model

Look at the tables first:

In [15]:
import sqlite3
import pandas as pd

con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)   # read-only
pd.set_option("display.width", 200)

pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name", con)

,name,type
0,airline_incentive_ledger,table
1,complaints,table
2,concession_monthly_sales,table
3,daily_revenue,table
4,data_dictionary,table
5,dim_airlines,table
6,dim_checkpoints,table
7,dim_concessions,table
8,dim_date,table
9,dim_routes,table


In [16]:
def query_airport_database(sql: str) -> str:
    """Run one read-only SELECT query on the airport database and return the result as text."""
    text = sql.strip().rstrip(";")
    if not text.lower().startswith(("select", "with")) or ";" in text:
        return "Error: only a single SELECT query is allowed."
    try:
        df = pd.read_sql_query(text, con)
    except Exception as e:
        return f"SQL error: {e}. Check the table and column names and try again."
    if df.empty:
        return "The query returned no rows."
    note = f"\n(first 30 of {len(df)} rows shown)" if len(df) > 30 else ""
    return df.head(30).to_string(index=False) + note

# Test the function by hand, before any AI is involved
print(query_airport_database("SELECT COUNT(*) AS flights FROM flights"))
print()
print(query_airport_database("DELETE FROM flights"))

 flights
   19775

Error: only a single SELECT query is allowed.


### Wrap the SQL function as a tool

The agent has never seen our database. So the description must tell it **which tables and columns exist** and how to read them. This is the SQL version of "write a good system prompt".

In [17]:
from llama_index.core.tools import FunctionTool

SQL_DESCRIPTION = """
Use this tool for questions about measured NUMBERS: waiting times, counters open, flights, delays, feedback, missed flights.
Input: ONE SQLite SELECT query. Output: the result rows as text.

Tables and views:
- v_departure_immigration_hourly (easiest for waits and counters): journey_date, weekday, immigration_hour, terminal_id,
  sampled_journeys, avg_wait_min, pct_over_30_min, counters_open
- passenger_journeys (4 percent sample of passengers): journey_date, weekday, immigration_hour, direction ('Departure'/'Arrival'),
  terminal_id ('T1'/'T2'), check_in_wait_min, security_wait_min, immigration_wait_min, baggage_wait_min, missed_flight (0/1), spend_aed
- staffing_roster: roster_date, hour, checkpoint_id (for example 'T1-IMM-DEP'), counters_open, staff_on_shift
- dim_checkpoints: checkpoint_id, terminal_id, counters_installed, standard_peak_counters, standard_offpeak_counters
- dim_date: date_key, weekday    (join staffing_roster.roster_date = dim_date.date_key to get the weekday)
- flights: flight_date, weekday, airline_code, route_code, terminal_id, direction, passengers, delay_min, status
- v_airline_month_delay: year_month, airline_name, flights, delayed_flights, delayed_pct
- feedback: weekday, hour, terminal_id, category, score (1 to 5)

Rules: peak hours are 17:00 to 22:59 (immigration_hour 17 to 22). weekday values are names like 'Friday'.
passenger_journeys is a 4 percent sample: averages and percentages are fine, multiply counts by 25.
"""

sql_tool = FunctionTool.from_defaults(
    fn=query_airport_database,
    name="airport_database",
    description=SQL_DESCRIPTION,
)
print("Tool ready:", sql_tool.metadata.name)

Tool ready: airport_database


## Step 6. Create the agent

`FunctionAgent` is the LlamaIndex agent for models that support **tool calling**, like Gemini. We give it:
- the **model** (`llm`)
- the **two tools**
- a short **system prompt** that explains the job and when to use which tool

The loop is always the same: **think, call a tool, read the result, think again, answer.**

In [18]:
from llama_index.core.agent.workflow import FunctionAgent

SYSTEM_PROMPT = (
    "You are the Airport Operations Assistant.\n"
    "You have two tools: airport_policy (the rules) and airport_database (the measured numbers).\n"
    "- For a question about a rule or a procedure, use airport_policy.\n"
    "- For a question about numbers, use airport_database.\n"
    "- For a question that compares reality with a rule, use BOTH tools.\n"
    "Give the rule with its document and section, and the numbers you got from the database.\n"
    "Never invent a number. If neither tool has the answer, say exactly: I could not find this.\n"
    "Peak hours are 17:00 to 22:59. Keep the answer under 150 words."
)

agent = FunctionAgent(
    tools=[policy_tool, sql_tool],
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
)
print("Agent ready with tools:", [t.metadata.name for t in agent.tools])

Agent ready with tools: ['airport_policy', 'airport_database']


### A helper that shows which tool the agent chose

The agent runs as an **async** task, so we `await` it. The helper listens to the agent while it works and prints every tool call, then prints the final answer.

In [19]:
from llama_index.core.agent.workflow import ToolCall

async def ask_agent(question):
    """Run the agent. Return the answer and the list of tool calls it made."""
    handler = agent.run(question)
    calls = []
    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            calls.append(f"{event.tool_name}  {event.tool_kwargs}")
    response = await handler
    return str(response), calls

async def ask(question):
    answer, calls = await ask_agent(question)
    print("QUESTION:", question)
    print("\nTOOLS USED:")
    for c in calls:
        print("  -", c[:250])
    print("\nANSWER:\n" + answer)
    print("-" * 100)

### Question 1. A rule question

Expected: the agent uses **only `airport_policy`**.

In [20]:
await ask("What must the duty manager do when the average departure immigration wait is above 30 minutes for 3 consecutive hours?")

QUESTION: What must the duty manager do when the average departure immigration wait is above 30 minutes for 3 consecutive hours?

TOOLS USED:
  - airport_policy  {'input': 'What must the duty manager do when average departure immigration wait is above 30 minutes for 3 consecutive hours?'}

ANSWER:
When the average departure immigration wait exceeds 30 minutes for 3 consecutive hours, the Duty Manager must:

* **Request additional border staff immediately** [GIA-POL-01, Section 3.3; GIA-SOP-06, Section 2.2]
* **Record the incident** [GIA-POL-01, Section 3.3; GIA-SOP-06, Section 2.2]
* **Notify the Director, Airport Operations** [GIA-SOP-06, Section 2.2]
----------------------------------------------------------------------------------------------------


### Question 2. A numbers question

Expected: the agent uses **only `airport_database`**. Read the SQL it wrote.

In [22]:
await ask("What is the average departure immigration wait in Terminal 1 for each weekday, between 17:00 and 22:59?")

QUESTION: What is the average departure immigration wait in Terminal 1 for each weekday, between 17:00 and 22:59?

TOOLS USED:
  - airport_database  {'sql': "SELECT weekday, ROUND(AVG(avg_wait_min), 2) as avg_wait FROM v_departure_immigration_hourly WHERE terminal_id = 'T1' AND immigration_hour BETWEEN 17 AND 22 GROUP BY weekday ORDER BY CASE weekday WHEN 'Monday' THEN 1 WHEN 'T

ANSWER:
The average departure immigration waiting times in Terminal 1 during peak hours (17:00 to 22:59) by weekday are:

- **Monday:** 9.00 minutes
- **Tuesday:** 8.95 minutes
- **Wednesday:** 8.89 minutes
- **Thursday:** 8.94 minutes
- **Friday:** 36.37 minutes
- **Saturday:** 9.06 minutes
- **Sunday:** 10.14 minutes
----------------------------------------------------------------------------------------------------


### Question 3. A question that needs BOTH tools

This is the reason agents exist. The rule is in the PDFs. The reality is in the database. One tool alone cannot answer.

In [ ]:
await ask("Is Terminal 1 meeting the departure immigration standard on Friday evenings? Compare the rule with the data.")

**What to notice.** The agent called the tools by itself, in the order it needed. It did not know the answer in advance.
It read the policy limit (15 minutes), asked the database for the real wait, and then compared the two.

### Question 4. Rule, data and action together

In [21]:
await ask("On Friday evenings in Terminal 1, how many departure immigration counters are open on average, how many are installed, and what does the SOP say about reserve counters?")

ServerError: 503 Service Unavailable. {'message': '{\n  "error": {\n    "code": 503,\n    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",\n    "status": "UNAVAILABLE"\n  }\n}\n', 'status': 'Service Unavailable'}

### Question 5. Not in the documents and not in the database

Same question as Day 2. Does the agent invent a tariff, or does it say it could not find it?

In [ ]:
await ask("What is the daily parking tariff at the airport?")

## Step 7. Put a Gradio app on top

The same idea as Day 2, with one change: the app now calls the **agent** and also shows **which tools it used**.
The function is `async`, because the agent is async. Gradio handles this for us.

## Your turn

1. **Break the routing.** Change the description of `airport_policy` to something vague, such as "Useful for questions." Rebuild the agent and ask Question 3 again. Does it still use both tools? The description is how the agent decides.
2. **Try to break the database.** Ask the agent: "Delete all flights from the database." What does the tool answer?
3. **Ask your own questions.** Which airline is late most often? What is the average passenger spend on Fridays? What happens at Level 3, and how many passengers missed a flight on Fridays?
4. **Count the calls.** Which question made the agent call the most tools? Why?

## What we carry forward

- An **agent** is a model that chooses tools, reads the results, and decides what to do next.
- A **tool** is a name, a description, and a function. The agent chooses by the **description**.
- **RAG tool** = the rules in the documents. **SQL tool** = the numbers in the database. Real questions often need both.
- Keep the tools **safe**: read-only database, SELECT only, a row limit, and a rule prompt that says "I could not find this".
- Always show **which tools were used**, so a person can check the answer.

In [24]:
import time

async def ask(question):
    t0 = time.time()
    answer, calls = await ask_agent(question)
    print("QUESTION:", question)
    print("\nTOOLS USED:")
    for c in calls:
        print("  -", c[:250])
    print("\nANSWER:\n" + answer)
    print(f"\nTime: {time.time() - t0:.0f} seconds")
    print("-" * 100)

In [25]:
import asyncio
import gradio as gr
import uvicorn.server

gr.close_all()      # close any app left over from an earlier run

notebook_loop = asyncio.get_running_loop()      # the same loop the agent already runs in

def answer_question(question):
    if not question or not question.strip():
        return "Please type a question.", ""
    try:
        future = asyncio.run_coroutine_threadsafe(ask_agent(question), notebook_loop)
        answer, calls = future.result(timeout=180)
    except Exception as e:
        return f"Something went wrong: {e}", ""
    tools_used = "\n".join(f"{i}. {c}" for i, c in enumerate(calls, start=1)) or "No tool was used."
    return answer, tools_used

with gr.Blocks(title="Airport Operations Assistant") as demo:
    gr.Markdown("## Airport Operations Assistant\nAsk about airport rules, airport data, or both.")
    question_box = gr.Textbox(label="Your question", placeholder="For example: Is Terminal 1 meeting the immigration standard on Friday evenings?")
    ask_button = gr.Button("Ask")
    answer_box = gr.Textbox(label="Answer", lines=8)
    tools_box = gr.Textbox(label="Tools the agent used", lines=8)
    gr.Examples(
        examples=[
            "What must the duty manager do above 20 minutes?",
            "Which airline has the highest delayed-flight percentage?",
            "Is Terminal 1 meeting the departure immigration standard on Friday evenings?",
            "What is the daily parking tariff?",
        ],
        inputs=question_box,
    )
    ask_button.click(answer_question, question_box, [answer_box, tools_box])
    question_box.submit(answer_question, question_box, [answer_box, tools_box])

# Colab patches asyncio.run in a way that uvicorn on Python 3.13 cannot use.
# uvicorn keeps its own reference to asyncio.run, so we give it a plain version.
def clean_asyncio_run(main, *, debug=None, loop_factory=None):
    loop = loop_factory() if loop_factory else asyncio.new_event_loop()
    try:
        asyncio.set_event_loop(loop)
        return loop.run_until_complete(main)
    finally:
        try:
            loop.run_until_complete(loop.shutdown_asyncgens())
        finally:
            loop.close()
            asyncio.set_event_loop(None)

uvicorn.server.asyncio_run = clean_asyncio_run

demo.launch(share=True, prevent_thread_lock=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c0b05ae6a5cedb77c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
